# ASTRA-GUARD: Phase 3 — Debris Collision Risk AI Exploration
**AI-Powered Integrated Space Mission Safety & Intelligence**
Team OMEN · AITHON 2.0 · Track 11 Aerospace, Telemetry & SpaceTech

This notebook demonstrates:
1. **SGP4 Analytical Orbit Propagation:** High-fidelity propagation of primary asset (Sentinel-2A) and tracked space debris (Fengyun-1C).
2. **3-Stage Conjunction Screening:** Coarse altitude envelope overlap, KD-Tree spatial screening, and sub-second TCA refinement.
3. **Foster 2-D Collision Probability ($P_c$):** B-plane projection and 2-D encounter integration (with screening-grade assumed covariance).
4. **XGBoost Kelvins CDM Classifier:** Machine learning risk severity classification and feature attributions.


In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))
from datetime import UTC, datetime

from ml.debris.module import DebrisRiskModule
from ml.debris.pc_engine import FosterCollisionProbabilityEngine
from ml.debris.propagator import SGP4Propagator
from ml.debris.screener import ConjunctionScreener

print('Environment and Debris AI modules loaded successfully.')

## 1. SGP4 Orbital Propagation & Physical Envelopes

In [2]:
s2a_tle1 = '1 40697U 15028A   26292.50000000  .00000100  00000-0  20000-4 0  9991'
s2a_tle2 = '2 40697  98.5700 125.0000 0001200  85.0000 275.1000 14.30820000500001'
fy1c_tle1 = '1 29834U 99025DF  26292.50000000  .00001500  00000-0  15000-3 0  9992'
fy1c_tle2 = '2 29834  98.6000 125.0200 0015000  90.0000 270.0000 14.30825000450002'

prim = SGP4Propagator.from_tle(s2a_tle1, s2a_tle2, name='SENTINEL-2A')
deb = SGP4Propagator.from_tle(fy1c_tle1, fy1c_tle2, name='FENGYUN-1C-DEB')

print('Primary Orbital Envelope:', prim.get_orbit_envelope())
print('Debris Orbital Envelope: ', deb.get_orbit_envelope())

## 2. 3-Stage Conjunction Screening & TCA Refinement

In [3]:
screener = ConjunctionScreener()
start_time = datetime(2026, 10, 20, 0, 0, 0, tzinfo=UTC)
conjunctions = screener.screen_catalog(prim, [deb], start_time, horizon_hours=48.0)
print(f'Detected Conjunctions: {len(conjunctions)}')
for c in conjunctions:
    print(f'  Encounter with {c["secondary_name"]}: TCA={c["tca"]}, Miss={c["miss_distance_m"]}m, RelVel={c["relative_velocity_kms"]} km/s')

## 3. Foster 2-D Collision Probability (Pc) Engine

In [4]:
pc_engine = FosterCollisionProbabilityEngine()
res_screening = pc_engine.compute_screening_pc(miss_distance_m=350.0)
print('Screening-grade Pc calculation (assumed covariance):')
print(f'  Pc:         {res_screening["pc"]:.2e}')
print(f'  Risk Class: {res_screening["risk_class"]}')
print(f'  Method:     {res_screening["pc_method"]} (Screening-Grade={res_screening["screening_grade"]})')

## 4. Debris Risk Module & Machine Learning Contract

In [5]:
module = DebrisRiskModule()
module.fit()
mod_res = module.score({'miss_distance_m': 350.0, 'pc': 2.4e-4, 'time_to_tca_hours': 12.0})
print('Debris ModuleResult:')
print(f'  Risk Score (r_d): {mod_res.score} (0..1)')
print(f'  Confidence:       {mod_res.confidence}')
print(f'  Critical Class:   {mod_res.evidence.get("risk_class")}')
print('\nFactor Attributions:')
attributions = module.explain({'miss_distance_m': 350.0, 'pc': 2.4e-4, 'time_to_tca_hours': 12.0})
for a in attributions:
    print(f'  - {a.driver:25s} contribution: {a.contribution * 100:.1f}% ({a.direction})')